# AgroConecta ACV — treinamento opcional no Google Colab

Notebook para reproduzir ou comparar experimentos de treinamento com aceleração por GPU. O modelo e as métricas publicados no repositório foram obtidos em uma execução distinta, registrada em `models/metadata.json` e `reports/test_metrics.json`. A execução deste notebook constitui um novo experimento e não altera retroativamente aquelas evidências.


## Ambiente e código-fonte
Nas configurações do Colab, selecionar um ambiente com GPU. A origem dos scripts e dados é o [repositório público](https://github.com/BernardoPRocha/agroconecta-acv).


In [ ]:
!git clone https://github.com/BernardoPRocha/agroconecta-acv.git
%cd /content/agroconecta-acv
%pip install -r requirements.txt

## Preparação de dados
O script prepara o recorte de classes do PlantDoc e cria os arquivos YOLO necessários ao treinamento.


In [ ]:
!python scripts/prepare_data.py

## Treinamento com GPU
Os parâmetros abaixo reproduzem a configuração nominal de 30 épocas e 320 pixels, mas diferenças de ambiente ou de aleatoriedade podem produzir resultados distintos. O melhor checkpoint é salvo em `models/best.pt` da cópia de trabalho do Colab.


In [ ]:
import torch
assert torch.cuda.is_available(), "A execução deste bloco requer GPU ativa no Colab."
print("GPU:", torch.cuda.get_device_name(0))
!python scripts/train.py --epochs 30 --imgsz 320 --batch 8 --device 0 --name experimento_colab

## Avaliação independente
A avaliação usa o conjunto de teste reservado. Os novos resultados devem ser tratados como pertencentes ao checkpoint treinado nesta execução.


In [ ]:
!python scripts/evaluate.py --device 0

## Exportação dos resultados
Os artefatos exportados incluem os pesos e relatórios produzidos pelo novo experimento.


In [ ]:
from google.colab import files
from pathlib import Path
import shutil

output = Path("/content/agroconecta_acv_experimento")
output.mkdir(exist_ok=True)
for directory in ("models", "reports", "runs"):
    source = Path(directory)
    if source.exists():
        shutil.copytree(source, output / directory, dirs_exist_ok=True)
shutil.make_archive(str(output), "zip", root_dir=output)
files.download(str(output) + ".zip")

## Interpretação
O conjunto de teste não deve ser utilizado repetidamente para selecionar hiperparâmetros. Comparações entre versões requerem identificação explícita dos pesos e das métricas de cada execução.
